### introduction to data parsing

In [3]:
import os
from typing import List,Dict,Any
import pandas as pd
import langchain

In [9]:

from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)


d:\RAG System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### understanding document structure

In [12]:
## create a simple document
doc =Document(
    page_content="This is a simple document. It contains some text that we will use for testing the text splitter.",
    metadata={"source": "test_document.txt",
              "author": "Essam",
              "date": "2024-06-01"}
)
print(doc.page_content)
print(doc.metadata)

This is a simple document. It contains some text that we will use for testing the text splitter.
{'source': 'test_document.txt', 'author': 'Essam', 'date': '2024-06-01'}


In [13]:
type(doc)

langchain_core.documents.base.Document

### Text file(.txt)

In [14]:
import os
os.makedirs("data/text_files", exist_ok=True)

In [16]:
sample__text ={
"data/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.


""",
"data/text_files/machine_learning.txt": """
Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    """



}

In [18]:
for file_path,content in sample__text.items():
    with open(file_path,"w", encoding="utf-8") as f:
        f.write(content)

### TextLoader load single file

In [26]:
from langchain_community.document_loaders import TextLoader
loader=TextLoader("data/text_files/python_intro.txt")
documents=loader.load()
print(documents[0].page_content)
print(documents[0].metadata)

Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.



{'source': 'data/text_files/python_intro.txt'}


### Directory loader multiple file text

In [27]:
from langchain_community.document_loaders import DirectoryLoader 
directory_loader=DirectoryLoader("data/text_files", glob="*.txt", show_progress=True,loader_cls=TextLoader,loader_kwargs={"encoding":"utf-8"})
documents=directory_loader.load()
for doc in documents:
    print("Content:")
    print(doc.page_content)
    print("Metadata:")
    print(doc.metadata)
    print("-"*50)

100%|██████████| 2/2 [00:00<00:00, 182.81it/s]

Content:

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
Metadata:
{'source': 'data\\text_files\\machine_learning.txt'}
--------------------------------------------------
Content:
Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to le

### text splitter -----> chunks

In [28]:
from langchain_text_splitters import(
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
text=documents[0].page_content
print(text)


Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    


In [30]:
char_splitter=CharacterTextSplitter(
    chunk_size=50,chunk_overlap=10,separator="\n",length_function=len
)
chuncks=char_splitter.split_text(text)
print("number of chunks:",len(chuncks))
for i,chunk in enumerate(chuncks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print("-"*30)

Created a chunk of size 97, which is longer than the specified 50
Created a chunk of size 95, which is longer than the specified 50
Created a chunk of size 56, which is longer than the specified 50
Created a chunk of size 60, which is longer than the specified 50
Created a chunk of size 65, which is longer than the specified 50
Created a chunk of size 85, which is longer than the specified 50


number of chunks: 9
Chunk 1:
Machine Learning Basics
------------------------------
Chunk 2:
Machine learning is a subset of artificial intelligence that enables systems to learn and improve
------------------------------
Chunk 3:
from experience without being explicitly programmed. It focuses on developing computer programs
------------------------------
Chunk 4:
that can access data and use it to learn for themselves.
------------------------------
Chunk 5:
Types of Machine Learning:
------------------------------
Chunk 6:
1. Supervised Learning: Learning with labeled data
------------------------------
Chunk 7:
2. Unsupervised Learning: Finding patterns in unlabeled data
------------------------------
Chunk 8:
3. Reinforcement Learning: Learning through rewards and penalties
------------------------------
Chunk 9:
Applications include image recognition, speech processing, and recommendation systems
------------------------------


#### recommended spliter

In [34]:
recursive_char_splitter=RecursiveCharacterTextSplitter(
    chunk_size=80,chunk_overlap=30,length_function=len,separators=[ ""]
)
chunks=recursive_char_splitter.split_text(text)
print("number of chunks:",len(chunks))
for i,chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print("-"*30)

number of chunks: 11
Chunk 1:
Machine Learning Basics

Machine learning is a subset of artificial intelligenc
------------------------------
Chunk 2:
bset of artificial intelligence that enables systems to learn and improve
from e
------------------------------
Chunk 3:
ms to learn and improve
from experience without being explicitly programmed. It
------------------------------
Chunk 4:
ing explicitly programmed. It focuses on developing computer programs
that can a
------------------------------
Chunk 5:
g computer programs
that can access data and use it to learn for themselves.

Ty
------------------------------
Chunk 6:
t to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: L
------------------------------
Chunk 7:
ing:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning
------------------------------
Chunk 8:
data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcem
------------------------------
Chunk 9:
n unl

In [35]:
token_splitter=TokenTextSplitter(
    chunk_size=20,chunk_overlap=5,length_function=lambda x: len(x.split())
)
chunks=token_splitter.split_text(text)
print("number of chunks:",len(chunks))
for i,chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print("-"*30)

number of chunks: 8
Chunk 1:

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and
------------------------------
Chunk 2:
 enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
------------------------------
Chunk 3:
 focuses on developing computer programs
that can access data and use it to learn for themselves.


------------------------------
Chunk 4:
 for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled
------------------------------
Chunk 5:
 Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled
------------------------------
Chunk 6:
 patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties


------------------------------
Chunk 7:
 rewards and penalties

Applications include image recognition, speech processing, and recommendatio

In [ ]:
from langchain_text_splitters import